# UMAP embedding and Leiden clustering (Fig. 2c–f)

Computes the pre-trained-ResNet UMAP embedding of time point 1 cancer and healthy cells and the
Leiden clustering shown in Figure 2, and saves them to `results/fig_2_umap_with_clusters.csv`.
The Figure 2 panels are plotted from that saved CSV in `figure_notebooks/figures_2_S2.ipynb`.

This notebook also contains the Leiden **silhouette-score tuning** sweep, which is only used to
pick the clustering resolution and is not shown in the paper.

> Run from the repository root so `data`/`eval`/`results` resolve. The final save cell overwrites
> the committed `results/fig_2_umap_with_clusters.csv` — only run it when you intend to regenerate it.

In [ ]:
from data import PlateDataset

import os
import torch
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import umap


device = 'cuda:0'

In [ ]:
data = PlateDataset([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], load_masks=True, load_dino=True)

## Pre-trained ResNet features

In [ ]:
from torchvision.models import resnet18 as make_resnet18
from torchvision.models.feature_extraction import create_feature_extractor
from torch.utils.data import DataLoader


def extract_resnet_patch_features(imgs, transform=None):
  model = make_resnet18(weights="DEFAULT").to(device)
  return_nodes = {
      'flatten': 'z',
  }
  feature_extractor = create_feature_extractor(model.eval().to(device), return_nodes=return_nodes)
  z = torch.zeros((len(imgs), 512))
  i = 0
  loader = DataLoader(imgs, batch_size=128, shuffle=False)
  for img_batch in tqdm(loader):
    img_batch = img_batch.to(device).repeat(1, 3, 1, 1)
    if transform is not None:
      img_batch = transform(img_batch)
    with torch.no_grad():
      z[i:i+len(img_batch)] = feature_extractor(img_batch)['z'].cpu()
    i += len(img_batch)
  return z


# ResNet feature extraction is GPU-nondeterministic; the cache keeps the embedding reproducible
# with the committed figure. Delete results/fig2_res_zs.npy to recompute the features from scratch.
_c = 'results/fig2_res_zs.npy'
if os.path.exists(_c):
    res_zs = torch.from_numpy(np.load(_c))
else:
    res_zs = extract_resnet_patch_features(data.imgs)
    np.save(_c, res_zs.numpy())
res_zs.shape

## UMAP embedding

Sub-sample up to 200 cells per cancer patient and 800 per healthy volunteer (time point 1 only),
then embed the ResNet features with UMAP.

In [ ]:
from umap.umap_ import UMAP  # python 3.9 issues

df = data.info.copy()

np.random.seed(12412)
vis_idx_c = []
pats = df['patient'].unique()
for p in pats:
    if p[0] == 'H':
        continue
    idx = df[(df['patient'] == p) & (df['time'] <= 1)].index
    if len(idx) > 200:
        vis_idx_c.append(np.random.choice(idx, size=200, replace=False))
print(len(vis_idx_c))

vis_idx_h = []
for p in pats:
    if p[0] == 'P':
        continue
    idx = df[(df['patient'] == p) & (df['time'] <= 1)].index
    if len(idx) > 800:
        vis_idx_h.append(np.random.choice(idx, size=800, replace=False))
print(len(vis_idx_h))
vis_idx = np.concatenate(vis_idx_c + vis_idx_h)
print(len(vis_idx))

np.random.shuffle(vis_idx)

feat_umap = UMAP(random_state=123).fit_transform(res_zs[vis_idx])
print(feat_umap.shape)

Assemble the dataframe of cell metadata + UMAP coordinates that the figure notebook reads.

In [ ]:
df = data.info.copy()
df['plate'] = df['plate'].astype(str)
df['diagnosis'] = df['patient'].str.startswith('H').replace({True : 'healthy', False : 'cancer T1'})
df.loc[df['diagnosis'] == 'healthy', 'plate healthy'] =  df.loc[df['diagnosis'] == 'healthy', 'plate']
df.loc[df['diagnosis'] == 'healthy', 'group'] = None
df[['umap_x', 'umap_y']] = pd.DataFrame(feat_umap, index=vis_idx)

# 'main group' column used by the Fig. 2f group scatter
main_groups = ['H&N cancer', 'Chordoma/Chondrosarcoma', 'CNS-Meningioma']
df['main group'] = None
df.loc[df['group'].isin(main_groups), 'main group'] = df.loc[df['group'].isin(main_groups), 'group']

## Leiden clustering and silhouette tuning

`cluster_leiden` sweeps a range of Leiden resolutions and reports a silhouette score for each.
This sweep is **only** used to choose the resolution and is not part of any paper figure.

In [ ]:
from eval import cluster_leiden

cluster_labels = cluster_leiden(np.array(res_zs)[vis_idx], df.loc[vis_idx])

Resolution 0.4 (6 clusters) is used for Figure 2 — chosen from the sweep above. The Leiden
integer ids are relabelled to `C-1`…`C-6` in descending cluster size. The Fig. 2c scatter and
the Fig. 2d montage are plotted in the figure notebook.

In [ ]:
# resolution 0.4 -> 6 clusters (chosen from the sweep above)
# label Leiden ids by descending cluster size, then map to the C-numbering used in the
# figures (C-1 purple, C-2 dark teal, C-3 brown, C-4 light teal, C-5 red, C-6 black)
lab = cluster_labels[0.4]
order = np.argsort(-np.bincount(lab))
label_order = ['C-5', 'C-1', 'C-4', 'C-3', 'C-2', 'C-6']  # by descending size
remap = {int(old): label_order[i] for i, old in enumerate(order)}
df['cluster'] = None
df.loc[vis_idx, 'cluster'] = [remap[v] for v in lab]
df['cluster'] = pd.Categorical(df['cluster'], categories=[f'C-{i}' for i in range(1, 7)])
df['cluster'].value_counts()

## Save

> Writing this file overwrites the committed UMAP/cluster CSV that Figure 2 is plotted from.

In [ ]:
df.to_csv('results/fig_2_umap_with_clusters.csv')